In [1]:
from dotenv import load_dotenv
load_dotenv("../.env")

True

In [2]:
import os

import pandas as pd
from sqlalchemy import create_engine

## Data Loading

In [3]:
dsn = os.environ.get("CORPUS_DSN")
db_conn = create_engine(dsn)

In [4]:
df_docs = pd.read_sql("SELECT * FROM documents", con=db_conn)
print(df_docs.shape)

df_docs.head()

(1583, 5)


,id,title,word_count,source_url,published_at
0,1bcfc529-9788-4f8b-a8e4-c2780922cb9e,Wakil Dubes Walanda Gumbira Ningali Holland In...,231,http://majalah-balebat.blogspot.com/2009/11/wa...,2009-11-16 00:00:00+00:00
1,7ad64ffc-449a-4218-b3fb-a9ad92925068,Warga Bogor Mapag Taun anyar Islam,265,http://majalah-balebat.blogspot.com/2009/12/wa...,2009-12-19 00:00:00+00:00
2,6c714fec-c629-4604-bfac-0096e1a0f624,Warugan Lemah: Pola Lembur Urang Sunda Buhun,1255,http://majalah-balebat.blogspot.com/2011/06/wa...,2011-06-12 00:00:00+00:00
3,5669e653-63ab-49b4-be39-7bf07f2eefe7,Manfaat Olahraga Pikeun Kasehatan,246,http://tipscaras.blogspot.com/2017/02/contoh-a...,2018-11-28 00:00:00+00:00
4,bfc35e44-876f-4ddd-a7c3-dbe6339a27f1,DINA JANDÉLA INDUNG,69,https://basasunda.com/puisi-bahasa-sunda,2023-07-24 00:00:00+00:00


In [5]:
df_docs_raw = pd.read_sql("SELECT * FROM documents_raw", con=db_conn)
print(df_docs_raw.shape)

df_docs_raw.head()

(1583, 5)


,id,parent_id,title,content,embedding
0,27fb5548-b366-457f-a274-05ccaf540b37,1bcfc529-9788-4f8b-a8e4-c2780922cb9e,Wakil Dubes Walanda Gumbira Ningali Holland In...,bogor hotél institut (bhi) gawé bareng jeung f...,"[-0.028707556,-0.0106562115,-0.0012308153,0.08..."
1,961cdd4c-aabc-4226-ade2-8330f6433dd8,7ad64ffc-449a-4218-b3fb-a9ad92925068,Warga Bogor Mapag Taun anyar Islam,bogor - datang ton anyar islam 1431-hijréh pap...,"[-0.019425172,0.0027185907,-0.012498547,0.0119..."
2,787867ce-ed3a-4ab8-b024-f7137032f55b,6c714fec-c629-4604-bfac-0096e1a0f624,Warugan Lemah: Pola Lembur Urang Sunda Buhun,naskah warugan lemah kandelna ngan tilu lempir...,"[-0.003110414,0.07159964,0.014448286,0.0208336..."
3,d217d76e-835f-4559-b391-239c175dba23,5669e653-63ab-49b4-be39-7bf07f2eefe7,Manfaat Olahraga Pikeun Kasehatan,anu ku urang tos terang olahraga teh penting p...,"[-0.037972152,0.057898182,0.022286188,0.014567..."
4,b19c2555-2143-4a42-ad71-051b156a0ba9,bfc35e44-876f-4ddd-a7c3-dbe6339a27f1,DINA JANDÉLA INDUNG,"méméh layung kubur panineungan dina jandéla, g...","[-0.012507847,-0.0128959995,0.016406883,0.0557..."


## EDA

In [6]:
dupe_title = df_docs["title"].value_counts()
dupe_title[dupe_title > 1]

title
Sawér Pangantén             5
HIJRAH NU PANUNGTUNG        4
LUKISAN                     3
Angklung                    3
Pépéling                    3
                           ..
HALIMUN DI TEGALAN EURIH    2
GARA-GARA HUJAN             2
NU PANASAN                  2
HALIMUN GANDRUNG            2
Kasenian Daerah Pasundan    2
Name: count, Length: 105, dtype: int64

In [7]:
dupe_title_url = df_docs[["title", "source_url"]].value_counts()
dupe_title_url[dupe_title_url > 1]

title                 source_url                                               
Pangabaran - 5        https://sundadigi.com/mantra/detail/88                       3
ANAK SI POLENG        https://sundadigi.com/fiksimini/chapter_fiksimini/115/939    2
Walungan Cikahuripan  https://sundadigi.com/sajak/detail/142                       2
ASBAK                 https://sundadigi.com/fiksimini/chapter_fiksimini/98/582     2
UBAR SONO             https://sundadigi.com/fiksimini/chapter_fiksimini/44/261     2
                                                                                  ..
BEURIT                https://sundadigi.com/fiksimini/chapter_fiksimini/73/1050    2
BONGAN                https://sundadigi.com/fiksimini/chapter_fiksimini/115/946    2
BANJIR                https://sundadigi.com/fiksimini/chapter_fiksimini/115/943    2
WARUNG PENGKOLAN      https://sundadigi.com/fiksimini/chapter_fiksimini/35/381     2
ABAH PALAY KURBAN     https://sundadigi.com/fiksimini/chapter_fiksimin

## Preprocessing

In [8]:
df_dedupe = df_docs.drop_duplicates(subset=["title", "source_url"], keep="last")
df_clean = df_dedupe.merge(df_docs_raw, left_on="id", right_on="parent_id")
df_clean = df_clean.drop(columns=["id_y", "title_y", "parent_id", "embedding"])
df_clean = df_clean.rename(columns={"id_x": "id", "title_x": "title"})
df_clean = df_clean[["id", "title", "content", "published_at", "word_count", "source_url"]]

df_clean

,id,title,content,published_at,word_count,source_url
0,1bcfc529-9788-4f8b-a8e4-c2780922cb9e,Wakil Dubes Walanda Gumbira Ningali Holland In...,bogor hotél institut (bhi) gawé bareng jeung f...,2009-11-16 00:00:00+00:00,231,http://majalah-balebat.blogspot.com/2009/11/wa...
1,7ad64ffc-449a-4218-b3fb-a9ad92925068,Warga Bogor Mapag Taun anyar Islam,bogor - datang ton anyar islam 1431-hijréh pap...,2009-12-19 00:00:00+00:00,265,http://majalah-balebat.blogspot.com/2009/12/wa...
2,6c714fec-c629-4604-bfac-0096e1a0f624,Warugan Lemah: Pola Lembur Urang Sunda Buhun,naskah warugan lemah kandelna ngan tilu lempir...,2011-06-12 00:00:00+00:00,1255,http://majalah-balebat.blogspot.com/2011/06/wa...
3,5669e653-63ab-49b4-be39-7bf07f2eefe7,Manfaat Olahraga Pikeun Kasehatan,anu ku urang tos terang olahraga teh penting p...,2018-11-28 00:00:00+00:00,246,http://tipscaras.blogspot.com/2017/02/contoh-a...
4,bfc35e44-876f-4ddd-a7c3-dbe6339a27f1,DINA JANDÉLA INDUNG,"méméh layung kubur panineungan dina jandéla, g...",2023-07-24 00:00:00+00:00,69,https://basasunda.com/puisi-bahasa-sunda
...,...,...,...,...,...,...
1494,221d26f2-6d2b-4088-86cd-fe01eaf52307,TINA KAMUS ISTILAH KESEHATAN DALAM KEBUDAYAAN ...,sababaraha cutat a abab acay ngacay aceng ngac...,2023-07-11 00:00:00+00:00,137,https://sundadigi.com/materi/detail/255
1495,4cc29dd6-fbdd-426a-8446-b50cc43d8f36,TINA NASKAH SUNDA BUHUN SANGHYANG SASANA MAHA ...,1 naka nihan ta muwah tangguh dasa ra ka nga i...,2023-07-11 00:00:00+00:00,962,https://sundadigi.com/materi/detail/254
1496,675405a0-1e3d-4bd3-8bf5-4dd623154bda,KEBON AWI 2,teu gugur teu angin catang badag rubuh halang ...,2023-09-13 00:00:00+00:00,88,https://sundadigi.com/fiksimini/chapter_fiksim...
1497,3b43e1da-2866-4270-825f-e9acfb401d88,v,babaléan diuk tina awi salon bangku diuk bésa ...,2023-05-03 00:00:00+00:00,218,https://sundadigi.com/materi/detail/66


In [9]:
df_clean.to_json("../data/corpus.jsonl", orient="records", lines=True)